<img align="right" src="images/tf.png" width="128"/>
<img align="right" src="images/logo.png" width="128"/>

# Tutorial

This notebook gets you started with using
[Text-Fabric](https://annotation.github.io/text-fabric/) for coding in the Hexapla.

Familiarity with the underlying
[data model](https://annotation.github.io/text-fabric/tf/about/datamodel.html)
is recommended.

## Installing Text-Fabric

See [here](https://annotation.github.io/text-fabric/tf/about/install.html)

## Tip
If you start computing with this tutorial, first copy its parent directory to somewhere else,
outside your repository.
If you pull changes from the repository later, your work will not be overwritten.
Where you put your tutorial directory is up to you.
It will work from any directory.

## Hexapla data

Text-Fabric will fetch a standard set of features for you from the newest GitHub release binaries.

The data will be stored in the `text-fabric-data` in your home directory.

# Features
The data of the corpus is organized in features.
They are *columns* of data.
Think of the corpus as a gigantic spreadsheet, where row 1 corresponds to the
first word, row 2 to the second word, and so on, for all ~ 180 K words,
followed by ~ 190 K nodes of other types.

The information which text each word has, constitutes a column in that spreadsheet.
The Hexapla corpus contains about 15 columns.

Instead of putting that information in one big table, the data is organized in separate columns.
We call those columns **features**.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import collections

# Incantation

The simplest way to get going is by this *incantation*:

In [ ]:
from tf.app import use

In [ ]:
A = use("rdtaylorjr/hexapla", hoist=globals())

You can see which features have been loaded, and if you click on a feature name, you find its documentation.
If you hover over a name, you see where the feature is located on your system.

## API

The result of the incantation is that we have a bunch of special variables at our disposal
that give us access to the text and data of the corpus.

At this point it is helpful to throw a quick glance at the text-fabric API documentation
(see the links under **API Members** above).

The most essential thing for now is that we can use `F` to access the data in the features
we've loaded.
But there is more, such as `N`, which helps us to walk over the text, as we see in a minute.

The **API members** above show you exactly which new names have been inserted in your namespace.
If you click on these names, you go to the API documentation for them.

# Counting

In order to get acquainted with the data, we start with the simple task of counting.

## Count all nodes
We use the
[`N.walk()` generator](https://annotation.github.io/text-fabric/tf/core/nodes.html#tf.core.nodes.Nodes.walk)
to walk through the nodes.

In [ ]:
A.indent(reset=True)
A.info("Counting nodes ...")

i = 0
for n in N.walk():
    i += 1

A.info("{} nodes".format(i))

## What are those nodes?
Every node has a type, like word, word group, stichos, verse.
But what exactly are they?

Text-Fabric has two special features, `otype` and `oslots`, that must occur in every Text-Fabric data set.
`otype` tells you for each node its type, and you can ask for the number of `slot`s in the text.

Here we go!

In [ ]:
F.otype.slotType

In [ ]:
F.otype.maxSlot

In [ ]:
F.otype.maxNode

In [ ]:
F.otype.all

In [ ]:
C.levels.data

This is interesting: above you see all the textual objects, with the average size of their objects,
the node where they start, and the node where they end.

## Count individual object types
This is an intuitive way to count the number of nodes in each type.
Note in passing, how we use the `indent` in conjunction with `info` to produce neat timed
and indented progress messages.

In [ ]:
A.indent(reset=True)
A.info("counting objects ...")

for otype in F.otype.all:
    i = 0

    A.indent(level=1, reset=True)

    for n in F.otype.s(otype):
        i += 1

    A.info("{:>7} {}s".format(i, otype))

A.indent(level=0)
A.info("Done")

# Texts

Every column of the Göttingen Hexapla Database is a *text*.
Here are the texts with the number of word groups (readings) and words in each.

In [ ]:
for t in F.otype.s("text"):
    print(
        f"{F.text.v(t):<12} {len(L.d(t, otype='word_group')):>6} readings"
        f" {len(L.d(t, otype='word')):>6} words"
    )

# Sections

The sections are *text*, *book*, and *chapter*.
Chapters and verses follow the Greek numbering of the database.

In [ ]:
v = T.nodeFromSection(("secunda", "Psalmi", 17))
T.sectionFromNode(v)

In [ ]:
verses = L.d(v, otype="verse")
len(verses)

In [ ]:
A.plain(verses[0])

# Feature statistics

`F`
gives access to all features.
Every feature has a method
`freqList()`
to generate a frequency list of its values, higher frequencies first.

In [ ]:
F.languageISO.freqList()

In [ ]:
F.reading.freqList()

The witnesses that attest the readings are in the feature `siglum`.
Here are the most frequent witnesses of the Secunda, Aquila, and Symmachus.

In [ ]:
for name in ("secunda", "aquila", "symmachus"):
    counts = collections.Counter()
    for g in F.otype.s("word_group"):
        if F.text.v(L.u(g, otype="text")[0]) == name and F.siglum.v(g):
            counts.update(F.siglum.v(g).split(","))
    print(name, counts.most_common(5))

# One position in all columns

A word group has a `rank`: its position within its stichos.
The verse `id` is the key of the passage in the database.
Word groups with the same verse `id`, stichos, and `rank` stand at the same position in all columns.

We collect the positions once, and then show the columns of a position.

In [ ]:
positions = collections.defaultdict(list)

for g in F.otype.s("word_group"):
    verse = L.u(g, otype="verse")[0]
    stichos = L.u(g, otype="stichos")[0]
    positions[(F.id.v(verse), F.rank.v(stichos), F.rank.v(g))].append(g)

len(positions)

In [ ]:
def column(position):
    for g in positions[position]:
        text = F.text.v(L.u(g, otype="text")[0])
        words = " ".join(F.text.v(w) or "" for w in L.d(g, otype="word"))
        print(f"{text:<12} {words}")


column((1025030006, 2, 1))

# The Secunda beside the MT

The Secunda gives the Hebrew text in Greek letters.
Here are the Secunda readings of a verse with the MT reading at the same position.

In [ ]:
def secunda(versekey):
    for (key, stichos, rank), groups in sorted(positions.items()):
        if key != versekey:
            continue
        texts = {F.text.v(L.u(g, otype="text")[0]): g for g in groups}
        if "secunda" in texts:
            sec = T.text(texts["secunda"]).strip()
            mt = T.text(texts["mt"]).strip() if "mt" in texts else ""
            print(f"{stichos} {rank:>2} {sec:<16} {mt}")


secunda(1025030006)

# Search

Text-Fabric has a search engine.
Here we look for Aquila readings that are attested by the Mercati palimpsest, Rahlfs 1098.

In [ ]:
query = """
text text=aquila
  word_group siglum~1098
"""
results = A.search(query)

In [ ]:
A.table(results, end=5)

And here for word groups in the Secunda that hold more than one reading at one position.

In [ ]:
query = """
text text=secunda
  word_group reading>0
"""
results = A.search(query)
A.table(results, end=5)

# All steps

* **start** your first step in mastering the Hexapla computationally
* **[search](search.ipynb)** turbo charge your hand-coding with search templates